In [1]:
import osmnx as ox
import pandas as pd
import numpy as np
import networkx as nx

In [7]:
G = ox.load_graphml("tests/data/test_rng.ml")
nodes = ox.graph_to_gdfs(G, edges=False, node_geometry=False)
nodes.head(2)

,y,x,street_count,highway
osmid,,,,
290700344,56.162667,93.360388,2,NaN
290700348,56.162327,93.359660,2,NaN


In [3]:
weight = 'highway'
weight in nodes.columns

True

In [11]:
weight = ['x', 'highway']
print(all(w in nodes.columns for w in weight))
weight = ['z', 'highway']
print(all(w in nodes.columns for w in weight))

True
False


In [20]:
from genesis.EAPP import metric

start_node = list(G.nodes())[2000]
routes = nx.single_source_dijkstra_path_length(G, start_node, weight='travel_time')
nx.set_node_attributes(G, pd.Series(routes), 'arrival_time')
# print(list(G.nodes(data=True))[:2])

# weight = 'arrival_time'
# nodes = ox.graph_to_gdfs(G, edges=False, node_geometry=False)
# np.max(nodes[weight])

max_time_t = 27.881282428571428
max_time = metric(metric_function=np.mean)(G)
print(max_time)

16.322393925534676


In [23]:
def custom_metric_function(s:pd.Series):
    return len(s[s<=10])/len(s)
    
start_node = list(G.nodes())[500]
routes = nx.single_source_dijkstra_path_length(G, start_node, weight='travel_time')
nx.set_node_attributes(G, routes, 'arrival_time')

val_t = 0.7136380123322452
val = metric(metric_function=custom_metric_function)(G)
val

0.7136380123322452

In [ ]:
G = ox.load_graphml("tests/data/test_rng.ml")

In [45]:
def estimated_arrival_time(G, 
                           path_function, 
                           sources, # target=None,
                           weight='travel_time', 
                           arrival_name='arrival_time',
                           nearest_name = 'nearest',
                           add_time=1,
                           cutoff=None):
    '''
    Алгоритм расчета времен прибытия в узлы графа из ближайшего стартового узла.

        # Аргументы
        `sources`:list|dict
            Стартовые узлы. Может быть списком узлов вида list(int), или словарем вида dict(int:str), где ключ - 
            идентификатор узла, значение - его наименование. Может использоваться для указания узлов в которых 
            расположены пожарные подразделения: {1234:'ПСЧ-1'}
        `path_function`: function
            Функция расчета кратчайших путей от единственного источника. 
            В качестве функции могут быть переданы реализации алгоритмов из пакета
            `networkx`. Например, реализация алгоритма Дейкстры: `nx.multi_source_dijkstra`.
            Пользователь может использовать собственные функции с
            интерфейсом `func(G: Graph, sources: Any, target: Any | None = None, cutoff: Any | None = None, 
            weight: str = "weight") -> (dict, dict)`
        `weight`:str или function  = "travel_time"
            Имя поля содержащего вес ребер, или функция позволяющая вычислять 
            вес динамически.
        `route_name`:str='route_time'
            Имя атрибута в котором будет сохранено время следования по маршруту
        `nearest_name` = 'nearest'
            Имя атрибута в котором будет сохранен идентификатор ближайшего стартового узла
        `cutoff`:int=None
            Ограничение расчета
    '''

    # Проверка типов входящих данных
    if isinstance(sources, list):
        if isinstance(sources, list):
            if len(sources)==0:
                raise ValueError('В sources нет ни одного элемента!')
            for element in sources:
                if not isinstance(element, int):
                    raise TypeError("Все идентификаторы узлов в списке sources должны иметь тип данных int!")
                if not element in G.nodes():
                    raise KeyError(f"Узел {element} отсутствует в графе G")
    elif isinstance(sources, dict):
        if len(sources)==0:
            raise ValueError('В sources нет ни одного элемента!')
        for node, name in sources.items():
            if not isinstance(name, str):
                raise TypeError("Все имена узлов в словаре sources должны иметь тип данных str!")
            if not node in G.nodes():
                raise KeyError(f"Узел {node} отсутствует в графе G")
    else:
        raise TypeError("Идентификатор узла должен иметь тип данных int или list(int)!")
    if not isinstance(G, nx.MultiDiGraph):
        raise TypeError("Аргумент G должен иметь тип nx.MultiDiGraph!")

    # Расчет
    # times, routes = nx.multi_source_dijkstra(G, sources=sources, cutoff=cutoff, weight=weight)
    times, routes = path_function(G, sources=sources, cutoff=cutoff, weight=weight)
    # if target:
    #     times = {k:v for k,v in times.items() if k in target}
    #     routes = {k:v for k,v in routes.items() if k in target}

    times = pd.Series(times)+add_time
    if isinstance(sources, dict):
        nearest = pd.Series({k:sources[route[0]] for k, route in routes.items()}, dtype=str)
    else:
        nearest = pd.Series({k:route[0] for k, route in routes.items()}, dtype='int64')

    # Установка результатов расчета в качестве атрибутов ребер
    nx.set_node_attributes(G, times, arrival_name)
    nx.set_node_attributes(G, nearest, nearest_name)

In [34]:
# sources = [list(G.nodes())[100], list(G.nodes())[1000]]
sources = {list(G.nodes())[100]:'ПСЧ-1',
           list(G.nodes())[1000]:'ПСЧ-2'}
target=list(G.nodes())[:1200] # Можно отбирать по вхождению в полигон

estimated_arrival_time(G, nx.multi_source_dijkstra, sources, target)
list(G.nodes(data=True))[:5]

[(290700344,
  {'y': 56.1626666,
   'x': 93.3603879,
   'street_count': 2,
   'arrival_time': 6.270903142857142,
   'nearest': 'ПСЧ-2'}),
 (290700348,
  {'y': 56.1623272,
   'x': 93.3596598,
   'street_count': 2,
   'arrival_time': 6.170113428571428,
   'nearest': 'ПСЧ-2'}),
 (290700352,
  {'y': 56.1615997,
   'x': 93.3581882,
   'street_count': 2,
   'arrival_time': 5.961232857142856,
   'nearest': 'ПСЧ-2'}),
 (290700353,
  {'y': 56.1611,
   'x': 93.3569672,
   'street_count': 2,
   'arrival_time': 5.800388285714285,
   'nearest': 'ПСЧ-2'}),
 (290700355,
  {'y': 56.1596152,
   'x': 93.3528134,
   'street_count': 2,
   'arrival_time': 5.276195714285714,
   'nearest': 'ПСЧ-2'})]

In [46]:
sources = {list(G.nodes())[100]:'ПСЧ-1',
           list(G.nodes())[1000]:'ПСЧ-2'}

estimated_arrival_time(G, nx.multi_source_dijkstra, sources)
list(G.nodes(data=True))[:5]

[(290700344,
  {'y': 56.1626666,
   'x': 93.3603879,
   'street_count': 2,
   'arrival_time': 6.270903142857142,
   'nearest': 'ПСЧ-2'}),
 (290700348,
  {'y': 56.1623272,
   'x': 93.3596598,
   'street_count': 2,
   'arrival_time': 6.170113428571428,
   'nearest': 'ПСЧ-2'}),
 (290700352,
  {'y': 56.1615997,
   'x': 93.3581882,
   'street_count': 2,
   'arrival_time': 5.961232857142856,
   'nearest': 'ПСЧ-2'}),
 (290700353,
  {'y': 56.1611,
   'x': 93.3569672,
   'street_count': 2,
   'arrival_time': 5.800388285714285,
   'nearest': 'ПСЧ-2'}),
 (290700355,
  {'y': 56.1596152,
   'x': 93.3528134,
   'street_count': 2,
   'arrival_time': 5.276195714285714,
   'nearest': 'ПСЧ-2'})]

In [53]:
from genesis.EAPP import metric

nodes = ox.graph_to_gdfs(G, edges=False, node_geometry=False)
# target=list(G.nodes())[:1200]
# nodes = nodes.loc[target]

print('Общая метрика', metric(metric_function=np.mean)(nodes), end='\n\n')

for unit in nodes['nearest'].unique():
    nodes_subset = nodes.query(f'nearest == "{unit}"')
    print(f'Метрика для {unit}', metric(metric_function=np.mean)(nodes_subset))

Общая метрика 6.397189738540803

Метрика для ПСЧ-2 6.806923127104616
Метрика для ПСЧ-1 6.055449428119199
Метрика для nan nan


In [1]:
import logging

In [2]:
logging.debug("A DEBUG Message")
logging.info("An INFO")
logging.warning("A WARNING")
logging.error("An ERROR")
logging.critical("A message of CRITICAL severity")

ERROR:root:An ERROR
CRITICAL:root:A message of CRITICAL severity


In [4]:
logging.basicConfig(level=logging.INFO, filename="py_log.log",filemode="w")
logging.debug("A DEBUG Message")
logging.info("An INFO")
logging.warning("A WARNING")
logging.error("An ERROR")
logging.critical("A message of CRITICAL severity")

ERROR:root:An ERROR
CRITICAL:root:A message of CRITICAL severity
